# GPT-2 Activation Patching: Two Exploratory Experiments

**Scope:** Educational, reproducible mechanistic-interpretability demonstrations on one France/Italy prompt pair. These experiments are **not** evidence of a discovered circuit or sociolinguistic-register representation.

**Experiment 1:** Patch the country-name token across layers.  
**Experiment 2:** Patch the final prompt token across layers.

Run all cells from a fresh Google Colab session (Python 3; a T4 GPU is sufficient). The results are written to two separate folders: `results/` and `results_final_position/`.


In [ ]:
%pip -q install "transformer-lens>=4,<5" matplotlib

## What the method does
- Compare two almost-identical prompts with different country names.
- Measure the model’s preference for the corresponding capital city.
- Replace **one internal activation** from one prompt with the activation from the other.
- Check whether this changes the model’s prediction.

An effective patch identifies a causally influential activation under the chosen intervention; it does **not**, alone, identify a complete neural circuit.

In [ ]:
"""Exploratory activation patching with TransformerLens 4.x and GPT-2 small.

Model downloads are required when running. This script deliberately makes no
claims about an outcome until you have inspected the empirical results.
"""

import csv
from pathlib import Path

import matplotlib.pyplot as plt
import torch

CLEAN = "The capital of France is"
CORRUPTED = "The capital of Italy is"
TARGET_CLEAN = " Paris"
TARGET_CORRUPTED = " Rome"


def single_token_id(model, text):
    """Return one continuation token ID; reject multi-token answers."""
    tokens = model.to_tokens(text, prepend_bos=False)
    if tokens.shape != (1, 1):
        raise ValueError(f"Expected one token for {text!r}; got {tokens.shape}")
    return int(tokens[0, 0])


def changed_token_position(clean_tokens, corrupted_tokens):
    """Ensure the prompts differ at exactly one token position."""
    if clean_tokens.shape != corrupted_tokens.shape:
        raise ValueError("Clean and corrupted prompts have different token lengths")
    changed = torch.where(clean_tokens[0] != corrupted_tokens[0])[0].tolist()
    if len(changed) != 1:
        raise ValueError(f"Expected one changed position, found {changed}")
    return int(changed[0])


def logit_difference(logits, positive_id, negative_id):
    """Unnormalized preference for positive over negative continuation."""
    return float((logits[0, -1, positive_id] - logits[0, -1, negative_id]).item())


def normalized_recovery(patched_score, corrupted_score, clean_score):
    """0 = corrupted baseline; 1 = clean baseline; values can exceed [0,1]."""
    denominator = clean_score - corrupted_score
    if abs(denominator) < 1e-6:
        raise ValueError("Behavioral baseline difference is too small for normalization")
    return (patched_score - corrupted_score) / denominator


def patch_position(model, base_tokens, source_cache, layer, position):
    """Replace a single residual-stream activation at one layer and position."""
    name = f"blocks.{layer}.hook_resid_post"
    source = source_cache[name]

    def patch(activation, hook):
        modified = activation.clone()
        modified[:, position, :] = source[:, position, :]
        return modified

    return model.run_with_hooks(base_tokens, fwd_hooks=[(name, patch)])


def main():
    from transformer_lens.model_bridge import TransformerBridge

    torch.manual_seed(0)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Loading GPT-2 small on {device} (first run downloads weights)...")
    model = TransformerBridge.boot_transformers("gpt2", device=device)
    model.eval()

    clean_tokens = model.to_tokens(CLEAN)
    corrupted_tokens = model.to_tokens(CORRUPTED)
    country_position = changed_token_position(clean_tokens, corrupted_tokens)
    if country_position == 0:
        raise ValueError("No unchanged prefix token available for a negative control")
    prefix_control_position = country_position - 1
    paris = single_token_id(model, TARGET_CLEAN)
    rome = single_token_id(model, TARGET_CORRUPTED)

    with torch.no_grad():
        names_filter = lambda name: name.endswith(".hook_resid_post")
        clean_logits, clean_cache = model.run_with_cache(clean_tokens, names_filter=names_filter)
        corrupted_logits, corrupted_cache = model.run_with_cache(
            corrupted_tokens, names_filter=names_filter
        )
        clean_score = logit_difference(clean_logits, paris, rome)
        corrupted_score = logit_difference(corrupted_logits, paris, rome)
        gap = clean_score - corrupted_score
        print(f"Clean Paris-minus-Rome logit difference:     {clean_score:+.3f}")
        print(f"Corrupted Paris-minus-Rome logit difference: {corrupted_score:+.3f}")
        print(f"Behavioral separation:                     {gap:+.3f}")
        if gap <= 0.5:
            raise RuntimeError(
                "Behavioral separation is too weak for this exploratory patching demo. "
                "Do not interpret or report a causal circuit. Check the chosen prompt pair."
            )

        rows = []
        for layer in range(model.cfg.n_layers):
            rescued_logits = patch_position(
                model, corrupted_tokens, clean_cache, layer, country_position
            )
            damaged_logits = patch_position(
                model, clean_tokens, corrupted_cache, layer, country_position
            )
            control_logits = patch_position(
                model, corrupted_tokens, clean_cache, layer, prefix_control_position
            )
            rescue_score = logit_difference(rescued_logits, paris, rome)
            damage_score = logit_difference(damaged_logits, paris, rome)
            control_score = logit_difference(control_logits, paris, rome)
            rows.append(
                {
                    "layer": layer,
                    "rescue_recovery": normalized_recovery(
                        rescue_score, corrupted_score, clean_score
                    ),
                    "reverse_damage": (clean_score - damage_score) / gap,
                    "unchanged_prefix_control": normalized_recovery(
                        control_score, corrupted_score, clean_score
                    ),
                }
            )

    result_dir = Path("results")
    result_dir.mkdir(exist_ok=True)
    with (result_dir / "patching_results.csv").open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    layers = [r["layer"] for r in rows]
    for field, label in [
        ("rescue_recovery", "Clean-to-corrupted rescue"),
        ("reverse_damage", "Corrupted-to-clean damage"),
        ("unchanged_prefix_control", "Unchanged-prefix control"),
    ]:
        plt.plot(layers, [r[field] for r in rows], marker="o", label=label)
    plt.axhline(0, linestyle="--", linewidth=0.8)
    plt.xlabel("Patched transformer layer")
    plt.ylabel("Normalized logit-difference effect")
    plt.title("Exploratory causal activation patching: France vs Italy")
    plt.legend()
    plt.tight_layout()
    plt.savefig(result_dir / "patching_results.png", dpi=170)
    print("Wrote results/patching_results.csv and results/patching_results.png")
    print("Exploratory example only; repeat on a preregistered set of prompt pairs.")


if __name__ == "__main__":
    main()


## Interpretation of Experiment 1

Country-token residual activation patching shows substantial early-layer rescue and almost no late-layer effect at this **earlier token position**. This is expected because late interventions at an earlier token cannot alter computations that already flowed toward the final prediction token.

The unchanged-prefix control should be zero: the prefix activations are identical across the clean and corrupted prompts, so this is only a basic implementation control.


## Experiment 2 — Final-token activation patching

Repeat the same intervention at the final prompt position. This tests how the importance of *where* we patch changes across layers. Note that replacing the entire final-layer residual vector at the final position trivially determines the readout, so a normalized effect of 1 there is expected by construction.


In [ ]:
# EXPERIMENT 2: Patch the final prompt position (not the country token).
# Uses helper functions defined in the Experiment 1 cell above.
# Saves outputs separately in results_final_position/.

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from transformer_lens.model_bridge import TransformerBridge

with torch.no_grad():
    torch.manual_seed(0)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = TransformerBridge.boot_transformers("gpt2", device=device)
    model.eval()

    clean_tokens = model.to_tokens(CLEAN)
    corrupted_tokens = model.to_tokens(CORRUPTED)
    country_position = changed_token_position(clean_tokens, corrupted_tokens)
    final_position = clean_tokens.shape[1] - 1
    if country_position == 0:
        raise ValueError("No unchanged prefix token available")
    prefix_position = country_position - 1

    paris = single_token_id(model, TARGET_CLEAN)
    rome = single_token_id(model, TARGET_CORRUPTED)

    names_filter = lambda name: name.endswith(".hook_resid_post")
    clean_logits, clean_cache = model.run_with_cache(clean_tokens, names_filter=names_filter)
    corrupted_logits, corrupted_cache = model.run_with_cache(
        corrupted_tokens, names_filter=names_filter
    )
    clean_score = logit_difference(clean_logits, paris, rome)
    corrupted_score = logit_difference(corrupted_logits, paris, rome)
    gap = clean_score - corrupted_score

    print(f"Clean score: {clean_score:+.3f}")
    print(f"Corrupted score: {corrupted_score:+.3f}")
    print(f"Behavioral separation: {gap:+.3f}")
    if gap <= 0.5:
        raise ValueError("Behavioral separation too small to interpret patching")

    rows = []
    for layer in range(model.cfg.n_layers):
        rescued = patch_position(model, corrupted_tokens, clean_cache, layer, final_position)
        damaged = patch_position(model, clean_tokens, corrupted_cache, layer, final_position)
        control = patch_position(model, corrupted_tokens, clean_cache, layer, prefix_position)
        rescue_score = logit_difference(rescued, paris, rome)
        damage_score = logit_difference(damaged, paris, rome)
        control_score = logit_difference(control, paris, rome)

        rows.append({
            "layer": layer,
            "rescue_recovery": normalized_recovery(rescue_score, corrupted_score, clean_score),
            "reverse_damage": (clean_score - damage_score) / gap,
            "unchanged_prefix_control": normalized_recovery(control_score, corrupted_score, clean_score),
        })

df = pd.DataFrame(rows)
display(df)

folder = Path("results_final_position")
folder.mkdir(exist_ok=True)
df.to_csv(folder / "patching_results.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 6))
for field, label in [
    ("rescue_recovery", "Clean-to-corrupted rescue"),
    ("reverse_damage", "Corrupted-to-clean damage"),
    ("unchanged_prefix_control", "Unchanged-prefix control"),
]:
    ax.plot(df["layer"], df[field], marker="o", label=label)
ax.axhline(0, linestyle="--", linewidth=0.8)
ax.set_xlabel("Transformer layer")
ax.set_ylabel("Normalized logit-difference effect")
ax.set_title("Experiment 2: Final-token activation patching")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(folder / "patching_results.png", dpi=170)
plt.show()
print("Experiment 2 completed; outputs saved to", folder)


## Limitations and next work

- These are exploratory demonstrations based on one prompt pair, not estimates across a population of prompts.
- The unchanged-prefix negative control is weak, because it patches identical activations.
- Layer-level residual patching cannot identify a unique head, neuron, feature, or circuit.
- Next: replicate across preregistered prompt pairs; implement stronger counterfactual controls; then study a new, explicitly exploratory extension of [Register Feature Family](https://github.com/Rinetta1981/register-feature-family), preserving the earlier failed behavioral-generalization result.
